# NLP Pipeline — Vehicle Verification System
### Natural Language Processing notebook demonstrating all concepts used
Covers: U1 (NLP pipeline, preprocessing), U2 (TF-IDF, embeddings), U3 (Classification), U4 (NER, IE), U5 (BERT)


In [ ]:
import sys
sys.path.insert(0, '../backend')

from nlp.text_cleaner import basic_cleanup, extract_plate_candidates, validate_indian_plate, full_nlp_preprocess
from nlp.plate_ner import extract_plate_with_ner, regex_ner, disambiguate_plate
from nlp.classifier import classify_plate, featurize_plate, vectorize_plate_tfidf
from nlp.bert_verifier import ocr_aware_distance, character_vector, cosine_similarity, verify_plate

print('NLP imports OK!')

## U1-T4: Text Cleanup and Preprocessing

In [ ]:
# Simulate messy OCR output
raw_texts = [
    'KA 01 AB 1234',
    'KA0IAB1Z34',      # OCR errors: 0I, Z
    'Vehicle: MH12CD5678 Reg',
    'KA-53-ST-9012',
]

for text in raw_texts:
    cleaned = basic_cleanup(text)
    candidates = extract_plate_candidates(cleaned)
    print(f'Raw: "{text}"')
    print(f'  Cleaned: "{cleaned}"')
    print(f'  Candidates: {candidates}')
    print()

## U4-T1 & U4-T2: Named Entity Recognition

In [ ]:
texts = [
    'The vehicle with plate KA01AB1234 requested access',
    'MH12CD5678 is trying to enter gate 2',
    'Detected number plate: KA 53 ST 9012',
]

for text in texts:
    entities = extract_plate_with_ner(text)
    print(f'Input: "{text}"')
    for e in entities:
        print(f'  Entity: [{e["label"]}] "{e["text"]}" (source: {e["source"]})')
    print()

## U2-T1 & U2-T2: TF-IDF Vectorization + Classification

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plates = ['KA01AB1234', 'KA01A1234', '22BH1234AA', 'MH04CD5678']

print('TF-IDF Classification Results:')
print('-' * 50)
for plate in plates:
    result = classify_plate(plate)
    print(f'Plate: {plate}')
    print(f'  Type: {result["vehicle_type"]} ({result["confidence"]*100:.0f}%)')
    print(f'  Features: {featurize_plate(plate)}')
    print()

## U2-T1: One-Hot Encoding & Character Vectors (U5-T2: Embeddings)

In [ ]:
p1 = 'KA01AB1234'
p2 = 'KA01AB1Z34'  # OCR error
p3 = 'MH12CD5678'  # Different plate

v1 = character_vector(p1)
v2 = character_vector(p2)
v3 = character_vector(p3)

print(f'Cosine similarity:')
print(f'  {p1} vs {p2} (OCR error): {cosine_similarity(v1, v2):.4f}')
print(f'  {p1} vs {p3} (different):  {cosine_similarity(v1, v3):.4f}')
print(f'  {p1} vs {p1} (same):       {cosine_similarity(v1, v1):.4f}')

# Visualize character vectors
chars = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')
fig, axes = plt.subplots(3, 1, figsize=(18, 8))
for ax, vec, plate in zip(axes, [v1, v2, v3], [p1, p2, p3]):
    ax.bar(range(len(chars)), vec, color='cyan', alpha=0.7)
    ax.set_xticks(range(len(chars)))
    ax.set_xticklabels(chars, fontsize=8)
    ax.set_title(f'Character Vector: {plate}')
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

## U5-T3 & U5-T4: BERT-style Verification Pipeline

In [ ]:
from nlp.bert_verifier import ocr_aware_distance, verify_plate

registered_plates = ['KA01AB1234', 'MH12CD5678', 'DL08EF9012', 'KA05BB9012']

# Test with OCR errors
test_cases = [
    ('KA01AB1234', 'Exact match'),
    ('KA0IAB1234', 'OCR confusion: 0I'),
    ('KA01AB123O', 'OCR confusion: 4→O'),
    ('XX99ZZ9999', 'Unregistered'),
]

print('Verification Results:')
print('=' * 60)
for plate, desc in test_cases:
    result = verify_plate(plate, registered_plates)
    print(f'Input: {plate} ({desc})')
    print(f'  Decision: {result["decision"]}')
    print(f'  Confidence: {result["confidence"]*100:.1f}%')
    print(f'  Matched:  {result.get("matched_plate", "—")}')
    print(f'  Method:   {result["method"]}')
    print()

## U1-T4: Full NLP Pipeline

In [ ]:
raw_ocr = 'KA 0I AB I234'  # Messy OCR output
result = full_nlp_preprocess(raw_ocr)

print('Full NLP Pipeline Output:')
print(f'  Raw OCR: "{result["raw"]}"')
print(f'  Cleaned: "{result["cleaned"]}"')
print(f'  Candidates: {result["candidates"]}')
print(f'  Best: {result["best_candidate"]}')